In [1]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v3" / "2015_2016"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2015_2016_raw_corrected.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2015_2016_raw_corrected.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2015_2016_raw_corrected.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2015_2016_raw_corrected.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2015_2016_raw_corrected.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file, dtype=str, keep_default_na=False)
labs = pd.read_csv(labs_file, dtype=str, keep_default_na=False)
questionnaire = pd.read_csv(questionnaire_file, dtype=str, keep_default_na=False)
Examination = pd.read_csv(Examination_file, dtype=str, keep_default_na=False)
Dietary = pd.read_csv(Dietary_file, dtype=str, keep_default_na=False)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2015_2016_corrected.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)


📊 Loaded Files:
Demographics: (9971, 49)
Labs: (9165, 34)
Questionnaire: (9971, 382)
Examination: (9544, 26)
Dietary: (9544, 248)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,9971,49,9971,0
1,labs,9165,34,9165,0
2,questionnaire,9971,382,9971,0
3,Examination,9544,26,9544,0
4,Dietary,9544,248,9544,0


📊 Final Unified Dataset Shape: (9971, 735)
Unique SEQN Count: 9971
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DR2TP184,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWS
0,83732,9,2,1,62,,3,3,1,,...,0.0,0.35,0.013,0.025,0.061,2,1440.0,480.0,960.0,2
1,83733,9,2,1,53,,3,3,1,,...,0.019,0.498,0.136,0.11699999999999999,0.489,2,480.0,0.0,480.0,4
2,83734,9,2,1,78,,3,3,2,,...,0.063,0.15100000000000002,0.009000000000000001,0.043,0.001,2,720.0,720.0,0.0,1
3,83735,9,2,2,56,,3,3,2,,...,,,,,,,,,,
4,83736,9,2,2,42,,4,4,2,,...,0.001,0.078,0.002,0.008,0.003,2,0.0,0.0,0.0,4
5,83737,9,2,2,72,,1,1,1,,...,0.0,0.096,0.0,0.005,0.027,2,480.0,480.0,0.0,99
6,83738,9,2,2,11,,1,1,1,141,...,0.001,0.133,0.009000000000000001,0.013,0.002,2,420.0,240.0,180.0,4
7,83739,9,2,1,4,,3,3,2,54,...,0.0,0.19799999999999998,0.003,0.016,0.04000000000000001,2,600.0,600.0,0.0,1
8,83740,9,2,1,1,13,2,2,2,14,...,0.0,0.0,0.0,0.0,0.0,2,135.0,135.0,0.0,99
9,83741,9,2,1,22,,4,4,1,,...,0.0,0.025,0.0,0.0,0.0,2,1014.0,0.0,1014.0,1



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_unified_2015_2016_corrected.csv
